In [ ]:
# Imports
import os, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Seeds
seed = 42
tf.random.set_seed(seed); np.random.seed(seed); random.seed(seed)

# Paths
DATA_DIR = pathlib.Path("../data")
TRAIN_DIR = DATA_DIR/"train"
VAL_DIR = DATA_DIR/"val"       # opcional, si existe
TEST_DIR = DATA_DIR/"test"     # opcional
MODEL_DIR = pathlib.Path("../models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Params
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 25
LR = 1e-3

# GPU check
print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# Dataset loaders
def make_ds(dir_path, with_labels=True, shuffle=True, subset=None, seed=42):
    # Create dataset
    return tf.keras.utils.image_dataset_from_directory(
        dir_path,
        labels='inferred' if with_labels else None,
        label_mode='int',
        color_mode='rgb',
        batch_size=BATCH,
        image_size=IMG_SIZE,
        shuffle=shuffle,
        seed=seed,
        subset=subset,
        validation_split=0.2 if subset else None
    )

# Build datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# Optional test
test_ds = make_ds(TEST_DIR, subset=None, shuffle=False) if TEST_DIR.exists() else None

# Class names
class_names = train_ds.class_names
num_classes = len(class_names)
class_names

In [ ]:
# Cache/prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
if test_ds: test_ds = test_ds.cache().prefetch(AUTOTUNE)

# Augment layer
data_augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augment")

In [ ]:
# Base model
base = keras.applications.EfficientNetB0(
    include_top=False, input_shape=IMG_SIZE+(3,), weights="imagenet"
)
base.trainable = False  # freeze base

# Head
inputs = layers.Input(shape=IMG_SIZE+(3,))
x = data_augment(inputs)
x = keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)                         # regularize
x = layers.Dense(256, activation="relu",
                 kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)                         # regularize
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="efficientnet_scenes")

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Callbacks
ckpt_path = str(MODEL_DIR/"best_model.keras")
callbacks = [
    keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy",
                                    save_best_only=True, mode="max"),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-6)
]

# Train frozen
hist1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=2
)

In [ ]:
# Unfreeze top
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

hist2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10, callbacks=callbacks, verbose=2
)

In [ ]:
# Load best
best = keras.models.load_model(ckpt_path)

# Val metrics
val_loss, val_acc = best.evaluate(val_ds, verbose=0)
print(f"Val acc: {val_acc:.4f}")

# Predictions val
y_true, y_pred = [], []
for imgs, labels in val_ds:
    probs = best.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(probs, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

# Report
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation='nearest')
ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.title("Confusion Matrix"); plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# Save final
final_path = MODEL_DIR/"best_model.keras"
best.save(final_path)  # formato .keras requerido
print("Saved:", final_path)

# Optional: export config
meta = {
    "img_size": IMG_SIZE, "classes": class_names,
    "val_accuracy": float(val_acc)
}
with open(MODEL_DIR/"model_meta.json","w") as f: json.dump(meta, f, indent=2)

In [ ]:
# Predict function
def predict_image(img_path, top_k=3):
    # Read img
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    x = keras.utils.img_to_array(img)
    x = np.expand_dims(x, 0)
    x = keras.applications.efficientnet.preprocess_input(x)
    # Predict
    probs = best.predict(x, verbose=0)[0]
    idxs = np.argsort(probs)[::-1][:top_k]
    return [(class_names[i], float(probs[i])) for i in idxs]

# Example
# predict_image("../some_image.jpg", top_k=3)